In [1]:
# =========================================
# IMPORT THƯ VIỆN
# =========================================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter, defaultdict
from scipy.sparse import lil_matrix
from tqdm import tqdm

print("All libraries imported successfully!")



All libraries imported successfully!


In [2]:
import pandas as pd
df=pd.read_csv("/kaggle/input/datasets/ldhhieu18/final-data/laodong_all_news_with_split.csv")

In [3]:
# =========================================
# CHUẨN BỊ DỮ LIỆU
# =========================================
print("\n" + "="*80)
print("PREPARING DATA FROM 'content' COLUMN (TRAIN ONLY)")
print("="*80)

# Lọc chỉ lấy tập train (is_test == 0)
df_train = df[df['is_test'] == 0].copy()
print(f"Total rows: {len(df)}")
print(f"Train rows (is_test=0): {len(df_train)}")
print(f"Test rows  (is_test=1): {len(df) - len(df_train)}")

texts = df_train['content'].tolist()
sentences = [text.split() for text in texts if isinstance(text, str) and len(text.strip()) > 0]
print(f"Example sentence: {sentences[0][:10]}")
print(f"Total sentences: {len(sentences)}")
print(f"Total tokens: {sum(len(s) for s in sentences)}")


PREPARING DATA FROM 'content' COLUMN (TRAIN ONLY)
Total rows: 6000
Train rows (is_test=0): 5100
Test rows  (is_test=1): 900
Example sentence: ['Lãi', 'suất', 'ngân', 'hàng', 'hôm', 'nay', '2.1:', 'Tăng', 'mạnh,', 'đua']
Total sentences: 5100
Total tokens: 2681617


The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [4]:
# =========================================
# XÂY DỰNG VOCABULARY
# =========================================
print("\n⏳ Building vocabulary...")

word_freq = Counter()
for sentence in sentences:
    word_freq.update(sentence)

# Tạo word2idx
word2idx = {'<PAD>': 0, '<UNK>': 1}
idx2word = {0: '<PAD>', 1: '<UNK>'}

idx = 2
for word, freq in word_freq.most_common():
    if freq >= 2:
        word2idx[word] = idx
        idx2word[idx] = word
        idx += 1

vocab_size = len(word2idx)
print(f"Vocabulary built: {vocab_size} words")




⏳ Building vocabulary...
Vocabulary built: 28126 words


In [5]:
# =========================================
# XÂY DỰNG CO-OCCURRENCE MATRIX
# =========================================
print("\n" + "="*80)
print("BUILDING CO-OCCURRENCE MATRIX")
print("="*80)

WINDOW_SIZE = 5

print(f"Building co-occurrence matrix (window={WINDOW_SIZE})...")

# Sử dụng sparse matrix để tiết kiệm bộ nhớ
cooccurrence_matrix = defaultdict(lambda: defaultdict(int))

# Build co-occurrence
for sentence in tqdm(sentences, desc="Processing sentences"):
    # Chuyển sang indices
    indices = [word2idx.get(word, word2idx['<UNK>']) for word in sentence]
    
    # Skip PAD và UNK
    indices = [idx for idx in indices if idx > 1]
    
    # Count co-occurrences
    for i, center_idx in enumerate(indices):
        # Define context window
        start = max(0, i - WINDOW_SIZE)
        end = min(len(indices), i + WINDOW_SIZE + 1)
        
        for j in range(start, end):
            if i != j:
                context_idx = indices[j]
                distance = abs(i - j)
                # Weight by distance (closer words have higher weight)
                weight = 1.0 / distance
                cooccurrence_matrix[center_idx][context_idx] += weight

print(f"Co-occurrence matrix built")
print(f"   Non-zero entries: {sum(len(v) for v in cooccurrence_matrix.values())}")




BUILDING CO-OCCURRENCE MATRIX
Building co-occurrence matrix (window=5)...


Processing sentences: 100%|██████████| 5100/5100 [00:20<00:00, 244.33it/s]

Co-occurrence matrix built
   Non-zero entries: 5161648


In [6]:
# =========================================
# GLOVE MODEL IMPLEMENTATION
# =========================================
print("\n" + "="*80)
print("IMPLEMENTING GLOVE MODEL")
print("="*80)

class GloVeModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        
        # Target word embeddings (W)
        self.target_embeddings = nn.Embedding(vocab_size, embedding_dim)
        # Context word embeddings (W_tilde)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)
        # Biases
        self.target_biases = nn.Embedding(vocab_size, 1)
        self.context_biases = nn.Embedding(vocab_size, 1)
        
        # Initialize
        nn.init.uniform_(self.target_embeddings.weight, -0.5, 0.5)
        nn.init.uniform_(self.context_embeddings.weight, -0.5, 0.5)
        nn.init.zeros_(self.target_biases.weight)
        nn.init.zeros_(self.context_biases.weight)
    
    def forward(self, target, context):
        # Get embeddings
        target_embed = self.target_embeddings(target)
        context_embed = self.context_embeddings(context)
        
        # Get biases
        target_bias = self.target_biases(target).squeeze()
        context_bias = self.context_biases(context).squeeze()
        
        # Compute dot product
        dot_product = (target_embed * context_embed).sum(dim=1)
        
        # Add biases
        output = dot_product + target_bias + context_bias
        
        return output

def weighting_function(x, x_max=100, alpha=0.75):
    """GloVe weighting function"""
    return torch.where(x < x_max, (x / x_max) ** alpha, torch.ones_like(x))




IMPLEMENTING GLOVE MODEL


In [7]:
# =========================================
# PREPARE TRAINING DATA
# =========================================
print("⏳ Preparing training data...")

train_data = []
for target_idx, contexts in cooccurrence_matrix.items():
    for context_idx, count in contexts.items():
        train_data.append((target_idx, context_idx, count))

print(f"Training data prepared: {len(train_data)} pairs")

# Convert to tensors
targets = torch.LongTensor([x[0] for x in train_data])
contexts = torch.LongTensor([x[1] for x in train_data])
counts = torch.FloatTensor([x[2] for x in train_data])



⏳ Preparing training data...
Training data prepared: 5161648 pairs


In [8]:
# =========================================
# TRAINING GLOVE
# =========================================
print("\n" + "="*80)
print("TRAINING GLOVE MODEL")
print("="*80)

EMBEDDING_DIM = 300
LEARNING_RATE = 0.05
EPOCHS = 50
BATCH_SIZE = 1024

# Initialize model
glove_model = GloVeModel(vocab_size, EMBEDDING_DIM)
optimizer = optim.Adagrad(glove_model.parameters(), lr=LEARNING_RATE)

# Create batches
num_batches = (len(train_data) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Training configuration:")
print(f"   Embedding dim: {EMBEDDING_DIM}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Total batches: {num_batches}")

# Training loop
for epoch in range(EPOCHS):
    total_loss = 0
    
    # Shuffle data
    perm = torch.randperm(len(targets))
    targets_shuffled = targets[perm]
    contexts_shuffled = contexts[perm]
    counts_shuffled = counts[perm]
    
    progress_bar = tqdm(range(num_batches), desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch_idx in progress_bar:
        start_idx = batch_idx * BATCH_SIZE
        end_idx = min((batch_idx + 1) * BATCH_SIZE, len(targets))
        
        batch_targets = targets_shuffled[start_idx:end_idx]
        batch_contexts = contexts_shuffled[start_idx:end_idx]
        batch_counts = counts_shuffled[start_idx:end_idx]
        
        # Forward pass
        predictions = glove_model(batch_targets, batch_contexts)
        
        # GloVe loss: weighted least squares
        weights = weighting_function(batch_counts)
        loss = (weights * (predictions - torch.log(batch_counts)) ** 2).mean()
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / num_batches
    print(f"Epoch {epoch+1}/{EPOCHS} - Average Loss: {avg_loss:.4f}")

print("\nGloVe training completed!")




TRAINING GLOVE MODEL
Training configuration:
   Embedding dim: 300
   Learning rate: 0.05
   Epochs: 50
   Batch size: 1024
   Total batches: 5041


Epoch 1/50: 100%|██████████| 5041/5041 [06:40<00:00, 12.59it/s, loss=0.1120]


Epoch 1/50 - Average Loss: 0.1463


Epoch 2/50: 100%|██████████| 5041/5041 [06:33<00:00, 12.82it/s, loss=0.0498]


Epoch 2/50 - Average Loss: 0.0549


...

Epoch 48/50 - Average Loss: 0.0074


Epoch 49/50: 100%|██████████| 5041/5041 [06:43<00:00, 12.50it/s, loss=0.0072]


Epoch 49/50 - Average Loss: 0.0074


Epoch 50/50: 100%|██████████| 5041/5041 [06:51<00:00, 12.25it/s, loss=0.0077]

Epoch 50/50 - Average Loss: 0.0073

GloVe training completed!


In [9]:
# =========================================
# EXTRACT EMBEDDINGS
# =========================================
print("\n🔗 Extracting GloVe embeddings...")

# GloVe uses sum of target + context embeddings
with torch.no_grad():
    embedding_matrix_glove = (
        glove_model.target_embeddings.weight + 
        glove_model.context_embeddings.weight
    ) / 2.0

print(f"Embedding matrix shape: {embedding_matrix_glove.shape}")




🔗 Extracting GloVe embeddings...
Embedding matrix shape: torch.Size([28126, 300])


In [10]:
# =========================================
# LƯU FILES
# =========================================
print("\n Saving files...")

torch.save(embedding_matrix_glove, "embedding_glove_content.pt")
torch.save(word2idx, "word2idx_glove_content.pt")
torch.save(idx2word, "idx2word_glove_content.pt")
torch.save(glove_model.state_dict(), "glove_model_state.pt")

print(" Saved files:")
print("   - embedding_glove_content.pt")
print("   - word2idx_glove_content.pt")
print("   - idx2word_glove_content.pt")
print("   - glove_model_state.pt")




 Saving files...
 Saved files:
   - embedding_glove_content.pt
   - word2idx_glove_content.pt
   - idx2word_glove_content.pt
   - glove_model_state.pt


In [11]:
# =========================================
# WORD SIMILARITY TEST
# =========================================
print("\n" + "="*80)
print("WORD SIMILARITY TEST")
print("="*80)

def find_similar_words(word, word2idx, embeddings, top_k=5):
    """Find most similar words using cosine similarity"""
    if word not in word2idx:
        return []
    
    word_idx = word2idx[word]
    word_embedding = embeddings[word_idx].unsqueeze(0)
    
    # Compute cosine similarity
    similarities = torch.nn.functional.cosine_similarity(
        word_embedding, embeddings
    )
    
    # Get top k
    top_indices = similarities.argsort(descending=True)[1:top_k+1]
    
    results = []
    for idx in top_indices:
        similar_word = idx2word[idx.item()]
        score = similarities[idx].item()
        results.append((similar_word, score))
    
    return results

# Test with sample words
sample_words = list(word2idx.keys())[2:7]  # Skip PAD and UNK
print(f"Sample words: {sample_words}")

for test_word in sample_words[:3]:
    similar = find_similar_words(test_word, word2idx, embedding_matrix_glove)
    print(f"\n Words similar to '{test_word}':")
    for word, score in similar:
        print(f"   • {word}: {score:.4f}")




WORD SIMILARITY TEST
Sample words: ['và', 'của', 'các', 'trong', 'có']

 Words similar to 'và':
   • của: 0.5988
   • các: 0.5720
   • đã: 0.5665
   • trong: 0.5571
   • có: 0.5470

 Words similar to 'của':
   • và: 0.5988
   • các: 0.5376
   • trong: 0.5290
   • cho: 0.5179
   • là: 0.5108

 Words similar to 'các':
   • một: 0.5744
   • và: 0.5720
   • với: 0.5573
   • việc: 0.5566
   • công: 0.5524


In [12]:
# =========================================
# LOAD VÀO MODEL
# =========================================
print("\n" + "="*80)
print(" LOADING INTO LSTM MODEL")
print("="*80)

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        out = self.fc(lstm_out[:, -1, :])
        return out

model = LSTMClassifier(vocab_size, EMBEDDING_DIM, 128, 3)

# Load GloVe embeddings
with torch.no_grad():
    model.embedding.weight.copy_(embedding_matrix_glove)

print(" GloVe embeddings loaded into LSTM model")
print(f"   Vocab size: {model.embedding.num_embeddings}")
print(f"   Embedding dim: {model.embedding.embedding_dim}")

print("\n" + "="*80)
print(" ALL DONE! GloVe embeddings trained from scratch!")
print("="*80)


 LOADING INTO LSTM MODEL
 GloVe embeddings loaded into LSTM model
   Vocab size: 28126
   Embedding dim: 300

 ALL DONE! GloVe embeddings trained from scratch!
